In [ ]:
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from statistics import median

import psycopg
import pymysql

pg_options = dict(host="127.0.0.1", port=55432, dbname="lab",
                  user="lab", password="lab", autocommit=True,
                  prepare_threshold=None)
mysql_options = dict(host="127.0.0.1", port=53306, database="lab",
                     user="lab", password="lab", autocommit=True)
pg_observer = psycopg.connect(**pg_options, application_name="observer")
mysql_observer = pymysql.connect(**mysql_options)
pg_observer.execute("SELECT 1").fetchone()
with mysql_observer.cursor() as cursor:
    cursor.execute("SELECT 1")
    cursor.fetchone()

def os_sample(service):
    output = subprocess.check_output([
        "docker", "exec", f"blog-connection-lab-{service}-1",
        "sh", "-c", """
        for f in /proc/[0-9]*/status; do
            awk '/^Name:/ {name=$2} /^Pid:/ {pid=$2}
                 /^Threads:/ {if (name=="postgres" || name=="mysqld")
                     print "task", pid, $2}' "$f" 2>/dev/null
        done
        cat /sys/fs/cgroup/memory.current
        cat /sys/fs/cgroup/memory.stat
        """
    ], text=True)
    tasks = [tuple(map(int, line.split()[1:]))
             for line in output.splitlines() if line.startswith("task ")]
    current = next(int(line) for line in output.splitlines() if line.isdigit())
    stat = dict(line.split() for line in output.splitlines()
                if not line.startswith("task ") and not line.isdigit())
    return dict(processes=len(tasks), threads=sum(t for _, t in tasks),
                tasks=tasks, memory_mib=current / 2**20,
                anon_mib=int(stat["anon"]) / 2**20)

def measure(service):
    samples = [os_sample(service) for _ in range(5)]
    result = samples[-1]
    result["memory_mib"] = median(s["memory_mib"] for s in samples)
    result["anon_mib"] = median(s["anon_mib"] for s in samples)
    return result

def show_change(name, before, after):
    print(f"{name}: processes {before['processes']} -> {after['processes']}, "
          f"OS threads {before['threads']} -> {after['threads']}")
    print(f"cgroup memory: {before['memory_mib']:.1f} -> "
          f"{after['memory_mib']:.1f} MiB "
          f"(delta {after['memory_mib'] - before['memory_mib']:+.1f})")
    print(f"anonymous memory: {before['anon_mib']:.1f} -> "
          f"{after['anon_mib']:.1f} MiB "
          f"(delta {after['anon_mib'] - before['anon_mib']:+.1f})")


In [ ]:
pg_before = measure("postgres")
pg_clients = [psycopg.connect(**pg_options, application_name="lab-direct")
              for _ in range(40)]
for connection in pg_clients:
    connection.execute("SELECT 1").fetchone()
time.sleep(2)
pg_after = measure("postgres")
idle = pg_observer.execute("""
    SELECT count(*) FROM pg_stat_activity
    WHERE application_name = 'lab-direct' AND state = 'idle'
""").fetchone()[0]
print("PostgreSQL idle application connections:", idle)
show_change("PostgreSQL", pg_before, pg_after)
print("Backend PID / OS threads:",
      [(c.info.backend_pid, dict(pg_after["tasks"])[c.info.backend_pid])
       for c in pg_clients[:3]])


In [ ]:
mysql_before = measure("mysql")
mysql_clients = [pymysql.connect(**mysql_options) for _ in range(40)]
for connection in mysql_clients:
    with connection.cursor() as cursor:
        cursor.execute("SELECT 1")
        cursor.fetchone()
time.sleep(2)
mysql_after = measure("mysql")
with mysql_observer.cursor() as cursor:
    cursor.execute("""
        SELECT count(*) FROM information_schema.PROCESSLIST
        WHERE USER = 'lab' AND ID <> CONNECTION_ID() AND COMMAND = 'Sleep'
    """)
    idle = cursor.fetchone()[0]
print("MySQL sleeping application connections:", idle)
show_change("MySQL", mysql_before, mysql_after)
print("mysqld PID / OS threads:", mysql_after["tasks"])


In [ ]:
for connection in pg_clients + mysql_clients:
    connection.close()
time.sleep(2)
print("PostgreSQL after closing:", measure("postgres")["processes"], "processes")
print("MySQL after closing:", measure("mysql")["threads"], "OS threads")
print("PostgreSQL:", pg_observer.execute("SHOW server_version").fetchone()[0])
with mysql_observer.cursor() as cursor:
    cursor.execute("SELECT VERSION(), @@thread_cache_size, @@thread_handling")
    print("MySQL version / thread cache / thread handling:", cursor.fetchone())
print(subprocess.check_output([
    "docker", "exec", "blog-connection-lab-pgbouncer-1", "pgbouncer", "--version"
], text=True).splitlines()[0])


In [ ]:
pool_options = dict(pg_options, port=56432)
pool_observer = psycopg.connect(**dict(pool_options, dbname="pgbouncer"))

def transaction(connection):
    return connection.execute(
        "SELECT pg_backend_pid(), pg_sleep(0.15)"
    ).fetchone()[0]

def run_transactions(connections):
    with ThreadPoolExecutor(max_workers=len(connections)) as executor:
        return list(executor.map(transaction, connections))

def pool_state():
    cursor = pool_observer.execute("SHOW CLIENTS")
    columns = [column.name for column in cursor.description]
    clients = [dict(zip(columns, row)) for row in cursor.fetchall()]
    client_count = sum(row["database"] == "lab" for row in clients)
    server_count = pg_observer.execute("""
        SELECT count(*) FROM pg_stat_activity
        WHERE backend_type = 'client backend'
          AND usename = 'lab' AND pid <> pg_backend_pid()
    """).fetchone()[0]
    return f"app clients={client_count}, PostgreSQL connections={server_count}"


In [ ]:
old_pod = [psycopg.connect(**pool_options) for _ in range(20)]
old_pids = set(run_transactions(old_pod))
print("Old pod only:", pool_state())
print("Backend PIDs used:", sorted(old_pids))

new_pod = [psycopg.connect(**pool_options) for _ in range(20)]
new_pids = set(run_transactions(new_pod))
print("Old + new pods:", pool_state())
print("Backend PIDs used:", sorted(new_pids))
print("New pod reused old backend PIDs:", new_pids == old_pids)

for connection in old_pod:
    connection.close()
print("New pod only:", pool_state())


In [ ]:
started = time.perf_counter()
completed = run_transactions(new_pod)
elapsed = time.perf_counter() - started
print("Completed transactions:", len(completed))
print("Distinct PostgreSQL backends:", len(set(completed)))
print("Each transaction holds its backend for 0.15 seconds")
print(f"Wall time for 20 concurrent requests: {elapsed:.2f} seconds")
print(pool_state())


In [ ]:
for connection in new_pod:
    connection.close()
pool_observer.close()
pg_observer.close()
mysql_observer.close()
